# Result model

### Caricamento dataset

Come al solito, recuperiamo il dataset completo.

In [59]:
import pandas as pd
import numpy as np

path = "static/da-result/result.csv"
dataset = pd.read_csv(path)

### Preparazione del target e split temporale

Procediamo:
- Mappando la colonna FTR in dati numerici (0 = Home, 1 = Draw, 2=Away)
- Ordinando cronologicamente il dataset
- Partizionandolo cronologicamente in set di training e set di test

In [60]:
from sklearn.metrics import accuracy_score, classification_report

# Mappo il target FTR in numeri
map = {'H': 0, 'D': 1, 'A': 2}
dataset['Numeric_target'] = dataset['FTR'].map(map)

# Ordino il dataset
dataset = dataset.sort_values(by='Date').reset_index(drop=True)

# Divido il traning set e test set in base alle stagioni (temporale)
X_train = dataset['Season'] < '2025-2026'
X_test = dataset['Season'] >= '2025-2026'

# Isolo il target per training e per test
Y_train = dataset.loc[X_train, 'Numeric_target']
Y_test = dataset.loc[X_test, 'Numeric_target']

print("====================================================")
print(f"DIMENSIONI DATASET")
print("====================================================")
print(f"Partite nel Training Set: {len(Y_train)}")
print(f"Partite nel Test Set: {len(Y_test)}")


DIMENSIONI DATASET
Partite nel Training Set: 1900
Partite nel Test Set: 380


### Random Baseline

Cominciamo osservando come un esperimento casuale si comporterebbe davanti alle partite del nostro test set (760). Questo passaggio lo utilizzamo per simulare lo scenario in cui un decisore (computer o un dado a tre facce), totalmente privo di dati storici o competenze calcistiche, tenti di indovinare l'esito dei match affidandosi al puro caso. 
L'output di questa simulazione fissa la nostra linea di partenza (random baseline) e diventa fondamentale per valutare il reale valore della ricerca.  

In [61]:
# Imposto il seed 
np.random.seed(42)

# Genero le predizioni totalmente casuali per le partite del test set
preds_casuali = np.random.choice([0, 1, 2], size=len(Y_test))

# Calcolo l'accuratezza del puro caso
accuratezza_puro_caso = accuracy_score(Y_test, preds_casuali)


print("\n==================================")
print("METRICHE DELL'ESPERIMENTO CASUALE")
print("==================================")
print(f"Accuratezza puro caso:    {accuratezza_puro_caso:.4f} ({accuratezza_puro_caso*100:.2f}%)")
print("\nReport dettagliato del Modello Casuale Puro:")
print(classification_report(Y_test, preds_casuali, target_names=['1', 'X', '2'], zero_division=0))


METRICHE DELL'ESPERIMENTO CASUALE
Accuratezza puro caso:    0.3211 (32.11%)

Report dettagliato del Modello Casuale Puro:
              precision    recall  f1-score   support

           1       0.40      0.34      0.37       148
           X       0.24      0.28      0.26        99
           2       0.33      0.32      0.32       133

    accuracy                           0.32       380
   macro avg       0.32      0.32      0.32       380
weighted avg       0.33      0.32      0.32       380

